# AST Chunking vs Naive Text Chunking

Comparison of:

- Naive fixed-length text splitting
- Tree-sitter AST chunking

Goal:
Show how syntax-aware chunking preserves semantic boundaries while naive chunking breaks functions and classes.

In [6]:
!pip install pandas

In [6]:
from pathlib import Path
import textwrap
import pandas as pd


In [8]:
import ast_chunker



ModuleNotFoundError: No module named 'ast_chunker'

In [3]:
def naive_split(text, chunk_size=300):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])

    return chunks

In [4]:
FILES = [
    "sample_repos\langchain\libs\text-splitters\langchain_text_splitters\base.py",
    "sample_repos\langchain\libs\text-splitters\langchain_text_splitters\json.py",
    "sample_repos\langchain\libs\text-splitters\langchain_text_splitters\latex.py",
    
]

<>:2: SyntaxWarning: invalid escape sequence '\l'
<>:3: SyntaxWarning: invalid escape sequence '\l'
<>:4: SyntaxWarning: invalid escape sequence '\l'
<>:2: SyntaxWarning: invalid escape sequence '\l'
<>:3: SyntaxWarning: invalid escape sequence '\l'
<>:4: SyntaxWarning: invalid escape sequence '\l'
C:\Users\Lavanya\AppData\Local\Temp\ipykernel_8752\615935026.py:2: SyntaxWarning: invalid escape sequence '\l'
  "sample_repos\langchain\libs\text-splitters\langchain_text_splitters\base.py",
C:\Users\Lavanya\AppData\Local\Temp\ipykernel_8752\615935026.py:3: SyntaxWarning: invalid escape sequence '\l'
  "sample_repos\langchain\libs\text-splitters\langchain_text_splitters\json.py",
C:\Users\Lavanya\AppData\Local\Temp\ipykernel_8752\615935026.py:4: SyntaxWarning: invalid escape sequence '\l'
  "sample_repos\langchain\libs\text-splitters\langchain_text_splitters\latex.py",


In [ ]:
comparison = []

for file in FILES:

    text = Path(file).read_text()

    naive = naive_split(text)

    ast_chunks = chunk_file(file)

    comparison.append({
        "file": Path(file).name,
        "naive_chunks": len(naive),
        "ast_chunks": len(ast_chunks)
    })

pd.DataFrame(comparison)

In [ ]:
text = Path(file).read_text()

naive = naive_split(text)

for i, chunk in enumerate(naive):

    print("="*80)
    print(f"Naive Chunk {i}")
    print("="*80)

    print(chunk)

In [ ]:
chunks = chunk_file(file)

for c in chunks:

    print("="*80)
    print(c.chunk_type, c.name)
    print("="*80)

    print(c.source)

In [ ]:
def looks_truncated(chunk):

    return (
        chunk.rstrip().endswith(",")
        or chunk.rstrip().endswith("(")
        or chunk.rstrip().endswith(":")
    )

In [ ]:
broken = sum(looks_truncated(c) for c in naive)

print(f"Broken chunks: {broken}")

In [ ]:
rows = []

for file in FILES:

    text = Path(file).read_text()

    naive = naive_split(text)

    ast_chunks = chunk_file(file)

    rows.append({

        "File": Path(file).name,

        "Naive Chunks": len(naive),

        "AST Chunks": len(ast_chunks),

        "Naive Broken?":
            any(looks_truncated(c) for c in naive),

        "AST Broken?":
            False
    })

pd.DataFrame(rows)